In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!pip install -q kaggle

In [5]:
!kaggle datasets list

ref                                                                 title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
amar5693/screen-time-sleep-and-stress-analysis-dataset              Screen Time, Sleep & Stress Analysis Dataset            787136  2026-02-13 06:56:18.757000          10219        197  1.0              
amar5693/student-performance-dataset                                Student Performance Dataset                             177286  2026-02-12 06:04:44.613000           8627        140  1.0              
aliiihussain/amazon-sales-dataset                                   Amazon_Sales_Dataset                                   1297759  2026-02-01 11:37:12.353000          10214        153

In [6]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [10]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

shl_audio_scoring_challenge_path = kagglehub.competition_download('shl-audio-scoring-challenge')

print('Data source import complete.', shl_audio_scoring_challenge_path)


Data source import complete. /root/.cache/kagglehub/competitions/shl-audio-scoring-challenge


In [8]:
# ============================================================
# SHL Audio Grammar Scoring - Advanced Pipeline v2
# Strategy: wav2vec2-large + handcrafted features + SVR/LGB/Ridge stack
# Optimized for Pearson correlation metric
# ============================================================
import os, warnings, gc
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
import torch
from torch.cuda.amp import autocast
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Find dataset paths
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))

CUDA available: True
GPU: Tesla T4


In [11]:
# Cell 2: Load data paths and CSVs
import glob

train_audio_dir = None
test_audio_dir = None
train_csv_path = None
test_csv_path = None

# for dirname, dirs, files in os.walk('/kaggle/input'):
#     for f in files:
#         fp = os.path.join(dirname, f)
#         if 'train.csv' in f and train_csv_path is None:
#             train_csv_path = fp
#         if 'test.csv' in f and test_csv_path is None:
#             test_csv_path = fp
#         if 'train' in dirname and any(f.endswith('.wav') or f.endswith('.mp3') for f in files):
#             if train_audio_dir is None:
#                 train_audio_dir = dirname
#         if 'test' in dirname and any(f.endswith('.wav') or f.endswith('.mp3') for f in files):
#             if test_audio_dir is None:
#                 test_audio_dir = dirname

train_csv_path = '/root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/csvs/train.csv'
test_csv_path = '/root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/csvs/test.csv'
train_audio_dir = '/root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/audios/train'
test_audio_dir =  '/root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/audios/test'

print('Train CSV:', train_csv_path)
print('Test CSV:', test_csv_path)
print('Train audio dir:', train_audio_dir)
print('Test audio dir:', test_audio_dir)

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print(train_df.head())
print(test_df.head())

# Auto-detect column names
filename_col = None
label_col = None
for col in train_df.columns:
    if any(x in col.lower() for x in ['file', 'audio', 'path', 'name']):
        filename_col = col
    if any(x in col.lower() for x in ['label', 'score', 'grammar', 'rating', 'grade']):
        label_col = col
if filename_col is None:
    filename_col = train_df.columns[0]
if label_col is None:
    label_col = train_df.columns[-1]
print(f'Using filename_col={filename_col}, label_col={label_col}')

Train CSV: /root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/csvs/train.csv
Test CSV: /root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/csvs/test.csv
Train audio dir: /root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/audios/train
Test audio dir: /root/.cache/kagglehub/competitions/shl-audio-scoring-challenge/dataset/audios/test
Train shape: (409, 2)
Test shape: (197, 1)
    filename  label
0  audio_173    3.0
1  audio_138    3.0
2  audio_127    2.0
3   audio_95    2.0
4   audio_73    3.5
    filename
0  audio_141
1  audio_114
2   audio_17
3   audio_76
4  audio_156
Using filename_col=filename, label_col=label


In [13]:
# Cell 3: Feature Extraction - wav2vec2-large + handcrafted audio features
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import librosa
from tqdm import tqdm

MODEL_NAME = 'facebook/wav2vec2-large-960h'
print(f'Loading {MODEL_NAME}...')
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
wav2vec_model = Wav2Vec2Model.from_pretrained(MODEL_NAME, output_hidden_states=True)
wav2vec_model = wav2vec_model.to(device)
wav2vec_model.eval()
print('Model loaded!')

def extract_handcrafted(audio, sr=16000):
    """Extract MFCC, spectral, and prosodic features."""
    feats = []
    # MFCCs (mean + std of 40 coefficients)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    feats.extend(mfcc.mean(axis=1).tolist())
    feats.extend(mfcc.std(axis=1).tolist())
    # Delta MFCCs
    delta_mfcc = librosa.feature.delta(mfcc)
    feats.extend(delta_mfcc.mean(axis=1).tolist())
    feats.extend(delta_mfcc.std(axis=1).tolist())
    # Spectral features
    spec_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    spec_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    spec_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)
    spec_flux = librosa.onset.onset_strength(y=audio, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(audio)
    rms = librosa.feature.rms(y=audio)
    for feat in [spec_centroid, spec_bandwidth, spec_rolloff, spec_flux.reshape(1,-1), zcr, rms]:
        feats.extend([feat.mean(), feat.std()])
    # Chroma features
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    feats.extend(chroma.mean(axis=1).tolist())
    feats.extend(chroma.std(axis=1).tolist())
    # Mel spectrogram stats
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=64)
    mel_db = librosa.power_to_db(mel)
    feats.extend([mel_db.mean(), mel_db.std(), mel_db.max(), mel_db.min()])
    # Pitch features (fundamental frequency)
    f0, voiced_flag, _ = librosa.pyin(audio, fmin=50, fmax=500, sr=sr)
    f0_valid = f0[voiced_flag] if voiced_flag is not None and voiced_flag.any() else np.array([0.0])
    feats.extend([np.nanmean(f0_valid), np.nanstd(f0_valid), np.sum(voiced_flag)/len(voiced_flag) if voiced_flag is not None else 0])
    # Speech rate proxy: number of onsets per second
    onsets = librosa.onset.onset_detect(y=audio, sr=sr)
    duration = len(audio) / sr
    feats.append(len(onsets) / (duration + 1e-8))
    return np.array(feats, dtype=np.float32)

def extract_features(audio_path, sr=16000, max_sec=30):
    """Extract wav2vec2 embeddings + handcrafted features."""
    try:
        audio, _ = librosa.load(audio_path, sr=sr)
        # Trim silence
        audio, _ = librosa.effects.trim(audio, top_db=20)
        max_len = sr * max_sec
        if len(audio) > max_len:
            audio = audio[:max_len]
        # Normalize amplitude
        if audio.std() > 0:
            audio = audio / (audio.std() + 1e-8)

        # --- wav2vec2 features: weighted average of ALL hidden layers ---
        inputs = processor(audio, sampling_rate=sr, return_tensors='pt', padding=True)
        input_values = inputs.input_values.to(device)
        with torch.no_grad():
            with autocast():
                outputs = wav2vec_model(input_values)
        hidden_states = outputs.hidden_states  # tuple of (batch, time, feat)
        # Weighted sum: higher weight on later layers
        num_layers = len(hidden_states)
        weights = np.linspace(0.5, 1.0, num_layers)
        weights = weights / weights.sum()
        weighted_hs = sum(w * hs for w, hs in zip(weights, hidden_states))
        mean_feat = weighted_hs.mean(dim=1).squeeze().cpu().float().numpy()
        std_feat = weighted_hs.std(dim=1).squeeze().cpu().float().numpy()
        # Concatenate mean and std
        w2v_feat = np.concatenate([mean_feat, std_feat])

        # --- Handcrafted features ---
        hc_feat = extract_handcrafted(audio, sr=sr)

        return np.concatenate([w2v_feat, hc_feat])
    except Exception as e:
        print(f'Error processing {audio_path}: {e}')
        # Return zeros as fallback (size = 2*1024 + handcrafted_dim)
        return np.zeros(2048 + 200, dtype=np.float32)

print('Feature extraction functions defined!')

Loading facebook/wav2vec2-large-960h...


Loading weights:   0%|          | 0/402 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!
Feature extraction functions defined!


In [14]:
# Cell 4: Extract train and test features
def resolve_audio_path(audio_dir, fname):
    """Try multiple extensions to find the audio file."""
    for ext in ['.wav', '.mp3', '']:
        candidate = os.path.join(audio_dir, fname if fname.endswith(ext) else fname + ext)
        if os.path.exists(candidate):
            return candidate
    return os.path.join(audio_dir, fname)

print('Extracting train features...')
train_features = []
train_labels = []
for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    fname = str(row[filename_col])
    audio_path = resolve_audio_path(train_audio_dir, fname)
    feats = extract_features(audio_path)
    train_features.append(feats)
    train_labels.append(float(row[label_col]))

X_train = np.array(train_features)
y_train = np.array(train_labels)
print(f'Train features shape: {X_train.shape}')
print(f'Label range: {y_train.min():.2f} - {y_train.max():.2f}, mean: {y_train.mean():.2f}')

print('Extracting test features...')
test_fname_col = filename_col if filename_col in test_df.columns else test_df.columns[0]
test_features = []
test_ids = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    fname = str(row[test_fname_col])
    audio_path = resolve_audio_path(test_audio_dir, fname)
    feats = extract_features(audio_path)
    test_features.append(feats)
    test_ids.append(row[test_df.columns[0]])

X_test = np.array(test_features)
print(f'Test features shape: {X_test.shape}')

# Free GPU memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Extracting train features...


100%|██████████| 409/409 [17:52<00:00,  2.62s/it]


Train features shape: (409, 2252)
Label range: 1.00 - 5.00, mean: 2.91
Extracting test features...


100%|██████████| 197/197 [08:13<00:00,  2.51s/it]


Test features shape: (197, 2252)


In [15]:
# Cell 5: Improved Stacked Ensemble - PCA + XGBoost + LightGBM + Ridge + SVR
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import lightgbm as lgb
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not available, skipping')

# Handle NaN/Inf in features
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

# --- Step 1: Scale + PCA ---
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# Use 300 PCA components (more than before for richer feature set)
n_components = min(300, X_tr_sc.shape[0] - 1, X_tr_sc.shape[1])
pca = PCA(n_components=n_components, random_state=SEED)
X_tr_pca = pca.fit_transform(X_tr_sc)
X_te_pca = pca.transform(X_te_sc)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.3f} with {n_components} components')

# --- Step 2: Train base models with 10-fold OOF predictions ---
N_FOLDS = 10
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

models = {
    'ridge': Ridge(alpha=10.0),
    'lgb': lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=6,
        num_leaves=50,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_samples=5,
        reg_alpha=0.05,
        reg_lambda=0.05,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    ),
    'svr': SVR(kernel='rbf', C=10.0, epsilon=0.1, gamma='scale'),
}
if HAS_XGB:
    models['xgb'] = xgb.XGBRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.05,
        reg_lambda=0.05,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0
    )

oof_preds = {name: np.zeros(len(y_train)) for name in models}
test_preds = {name: np.zeros(len(X_test)) for name in models}

for name, model in models.items():
    print(f'\nTraining {name}...')
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr_pca)):
        X_f, X_v = X_tr_pca[tr_idx], X_tr_pca[val_idx]
        y_f, y_v = y_train[tr_idx], y_train[val_idx]
        if name == 'lgb':
            model.fit(X_f, y_f, eval_set=[(X_v, y_v)],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
        elif name == 'xgb':
            model.fit(X_f, y_f, eval_set=[(X_v, y_v)], verbose=False)
        else:
            model.fit(X_f, y_f)
        oof_preds[name][val_idx] = model.predict(X_v)
        test_preds[name] += model.predict(X_te_pca) / N_FOLDS
        fold_rmse = np.sqrt(mean_squared_error(y_v, oof_preds[name][val_idx]))
        fold_r, _ = pearsonr(y_v, oof_preds[name][val_idx])
        print(f'  Fold {fold+1} RMSE: {fold_rmse:.4f}, Pearson r: {fold_r:.4f}')
    oof_rmse = np.sqrt(mean_squared_error(y_train, oof_preds[name]))
    oof_r, _ = pearsonr(y_train, oof_preds[name])
    print(f'{name} OOF RMSE: {oof_rmse:.4f}, OOF Pearson r: {oof_r:.4f}')

# --- Step 3: Meta-learner stacking (Ridge on OOF) ---
stacking_train = np.column_stack([oof_preds[n] for n in models])
stacking_test = np.column_stack([test_preds[n] for n in models])
meta = Ridge(alpha=1.0)
meta.fit(stacking_train, y_train)
final_pred = meta.predict(stacking_test)
meta_oof = meta.predict(stacking_train)
meta_oof_rmse = np.sqrt(mean_squared_error(y_train, meta_oof))
meta_oof_r, _ = pearsonr(y_train, meta_oof)
print(f'\nMeta-learner OOF RMSE: {meta_oof_rmse:.4f}')
print(f'Meta-learner OOF Pearson r: {meta_oof_r:.4f}')
print(f'Model weights: {dict(zip(models.keys(), meta.coef_))}')

PCA variance explained: 0.975 with 300 components

Training ridge...
  Fold 1 RMSE: 1.2029, Pearson r: 0.3094
  Fold 2 RMSE: 1.4768, Pearson r: 0.3323
  Fold 3 RMSE: 1.3739, Pearson r: 0.1329
  Fold 4 RMSE: 1.0774, Pearson r: 0.2979
  Fold 5 RMSE: 0.9949, Pearson r: 0.5930
  Fold 6 RMSE: 1.5112, Pearson r: -0.1629
  Fold 7 RMSE: 1.3378, Pearson r: 0.2651
  Fold 8 RMSE: 1.1033, Pearson r: 0.2964
  Fold 9 RMSE: 1.3632, Pearson r: 0.2439
  Fold 10 RMSE: 1.3060, Pearson r: 0.2819
ridge OOF RMSE: 1.2853, OOF Pearson r: 0.2609

Training lgb...
  Fold 1 RMSE: 0.5342, Pearson r: 0.5039
  Fold 2 RMSE: 0.7180, Pearson r: 0.6672
  Fold 3 RMSE: 0.6402, Pearson r: 0.5842
  Fold 4 RMSE: 0.6386, Pearson r: 0.1494
  Fold 5 RMSE: 0.7418, Pearson r: 0.7649
  Fold 6 RMSE: 0.5805, Pearson r: 0.5155
  Fold 7 RMSE: 0.5828, Pearson r: 0.5436
  Fold 8 RMSE: 0.6794, Pearson r: 0.3205
  Fold 9 RMSE: 0.6601, Pearson r: 0.4394
  Fold 10 RMSE: 0.7291, Pearson r: 0.4588
lgb OOF RMSE: 0.6536, OOF Pearson r: 0.5319



In [18]:
# Cell 6: Create submission file
from scipy.stats import pearsonr

submit_id_col = test_df.columns[0]

# Clip to training label range
y_min, y_max = y_train.min(), y_train.max()
final_pred_clipped = np.clip(final_pred, y_min, y_max)

# Do NOT round to integers - keep continuous for better Pearson correlation
# Only round if competition explicitly uses integer scores AND metric requires it
# final_pred_clipped = np.round(final_pred_clipped).astype(int)  # Disabled

submission = pd.DataFrame({
    submit_id_col: test_ids,
    label_col: final_pred_clipped
})
submission.to_csv('submission.csv', index=False)
print('Submission saved to submission.csv')
print(submission.head(10))
print(f'Submission shape: {submission.shape}')
print(f'Prediction stats - mean: {final_pred_clipped.mean():.3f}, std: {final_pred_clipped.std():.3f}')
print(f'Prediction range: {final_pred_clipped.min():.3f} - {final_pred_clipped.max():.3f}')

# OOF Pearson correlation
train_oof_final = meta.predict(stacking_train)
if len(np.unique(y_train)) > 1:
    r, _ = pearsonr(y_train, train_oof_final)
    print(f'OOF Pearson r: {r:.4f}')

Submission saved to submission.csv
      filename     label
0    audio_141  2.308771
1    audio_114  4.309154
2     audio_17  2.990069
3     audio_76  5.000000
4    audio_156  3.080942
5   audio_13_1  4.945381
6     audio_70  2.474768
7     audio_56  3.751875
8     audio_19  3.341041
9  audio_158_1  3.433821
Submission shape: (197, 2)
Prediction stats - mean: 3.337, std: 0.866
Prediction range: 1.633 - 5.000
OOF Pearson r: 0.6780


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
